In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from pydantic import BaseModel 
from langchain_groq import ChatGroq
from dotenv import load_dotenv


d:\GenAI\LangraphPlaylist\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [47]:
model = ChatGroq(
    model='openai/gpt-oss-20b',
    max_tokens=250
)

In [48]:
from pydantic import Field
class SentimentSchema(BaseModel):
    sentiment: Literal["positive","negative"] = Field(description="Sentiment of the review")
    


In [49]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal[
        "UX",
        "Performance",
        "Bug",
        "Feature Request",
        "Security",
        "Billing",
        "Account",
        "Compatibility",
        "Documentation",
        "Integration",
        "Data"
    ] = Field(
        description="The primary category of the customer's issue."
    )

    tone: Literal[
        "angry",
        "frustrated",
        "neutral",
        "confused",
        "satisfied",
        "disappointed",
        "urgent"
    ] = Field(
        description="The emotional tone expressed by the customer."
    )

    urgency: Literal[
        "low",
        "medium",
        "high",
        "critical"
    ] = Field(
        description="How urgently the issue should be addressed."
    )

   

In [20]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)


In [15]:
prompt = 'What is the sentiment of the following review -  Software is too bad.'
structured_model.invoke(prompt).sentiment

'negative'

In [16]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive","negative"]
    diagnosis: dict
    response: str
    

In [57]:
def find_sentiment(state: ReviewState):
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response","run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

def positive_response(state: ReviewState):
        prompt = f"""
            Write a warm thank-you message in response to the review
            \n\n {state['review']}
            Also Kindly ask the user to leave a  feedback on our website.
        """
        response = model.invoke(prompt).content
        return {'response': response}

def run_diagnosis(state: ReviewState):
    prompt = f"""
        Diagnose this negative review: \n\n {state['review']}\n
        Return issue_type, tone and urgency.
    """
    response = structured_model2.invoke(prompt)
    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):
        diagnosis = state['diagnosis']
        prompt = f"""
            You are a support assistant.
            The user had a {diagnosis['issue_type']} issue, sounded {diagnosis['tone']}, and marked urgence
            Write an empathetic, helpful resolution message.
        """
        response = model.invoke(prompt).content
        return {'response': response}


In [ ]:
graph = StateGraph(ReviewState)
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END) # (Assuming you also want this to go to END afterwards)


In [45]:
workflow = graph.compile()

In [58]:
initial_state = {
    'review': 'I have been using the app for about a month your app is very expensive and time wasting i will tell others to not to use this app'      
}
workflow.invoke(initial_state)

{'review': 'I have been using the app for about a month your app is very expensive and time wasting i will tell others to not to use this app',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Billing', 'tone': 'angry', 'urgency': 'medium'},
 'response': 'Hi there,\n\nI’m really sorry you’re dealing with this billing issue—thank you for bringing it to our attention and for letting us know how urgent it is. I understand how frustrating it can be when something doesn’t line up with what you expected.\n\nHere’s what I’m going to do right away:\n\n1. **Double‑check the invoice** – I’ll pull up your account and compare the charges with the plan you’re on.  \n2. **Re‑issue the correct invoice** – If there’s a discrepancy, I’ll generate a new, accurate invoice and send it to you within the next 30 minutes.  \n3. **Refund the over‑charge** – If you were over‑billed, I’ll initiate a refund that should appear in your original payment method within 3‑5'}